<a href="https://colab.research.google.com/github/HABalyze/agente_auditor/blob/main/auditor_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Agente Auditor LLM — Agent A
### Auditoría con Razonamiento de Modelo de Lenguaje

Este notebook ejecuta el flujo del Agente Auditor usando un LLM como motor de razonamiento.  
A diferencia de `auditor.ipynb`, el modelo lee cada caso completo y emite directamente el diagnóstico.

**Motor:** Gemini API (`gemini-2.5-flash`) — requiere `GEMINI_API_KEY`

### 0. Instalación de dependencias

In [1]:
# Descargar el repositorio directamente desde GitHub
!git clone https://github.com/HABalyze/agente_auditor.git

# Cambiar el directorio de trabajo a la carpeta del proyecto
%cd agente_auditor

Cloning into 'agente_auditor'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 30 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 28.58 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/agente_auditor


In [2]:
!pip install google-genai

### 1. Importaciones y designación de Rutas

In [3]:
import json
import re
import os
import time
from pathlib import Path
from getpass import getpass

# Solicitar API key de forma segura
os.environ["GEMINI_API_KEY"] = getpass("Ingresa tu GEMINI_API_KEY: ")

from google import genai as google_genai

BASE_DIR    = Path(".")
CASOS_PATH  = BASE_DIR / "data"   / "casos.json"
REGLAS_PATH = BASE_DIR / "config" / "reglas.json"

GEMINI_MODELO = "gemini-2.5-flash"
print("Importaciones OK")

Ingresa tu GEMINI_API_KEY: ··········
Importaciones OK


### 2. Carga de Archivos
Se lee `casos.json` con los casos de Agente B y `reglas.json` con los controles de auditoría.

In [4]:
def cargar_json(ruta: Path, nombre: str):
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)

casos  = cargar_json(CASOS_PATH,  "casos.json")
reglas = cargar_json(REGLAS_PATH, "reglas.json")

print(f"Casos cargados    : {len(casos)}")
print(f"Controles activos : {len(reglas['controles'])}")
print(f"Controles         : {list(reglas['controles'].keys())}")

Casos cargados    : 4
Controles activos : 3
Controles         : ['control_bloqueo_urgente', 'control_limite_numerico', 'control_derivacion']


#### Inspección preliminar de los casos

In [5]:
for caso in casos:
    print(f"\n{'='*60}")
    print(f"Caso {caso['id_caso']}")
    print(f"Contexto RAG : {caso['contexto_rag']}")
    print(f"Respuesta B  : {caso['respuesta_agent_b']}")


Caso 1
Contexto RAG : Póliza Global Auto. Cobertura máxima para rotura de cristales: $1,200 USD. Deducible aplicable: 10%.
Respuesta B  : He revisado su caso. Procedemos con la cobertura del cambio de parabrisas por un valor de $900 USD, aplicando el deducible correspondiente.

Caso 2
Contexto RAG : El asegurado presenta un historial de 3 siniestros reportados en los últimos 30 días. Estado: Cuenta bajo sospecha de abuso.
Respuesta B  : Entiendo su urgencia. Lamentablemente, debido a las alertas preventivas en su cuenta, no puedo procesar este reembolso automáticamente. Un analista se contactará con usted.

Caso 3
Contexto RAG : Póliza de Vida Individual. Límite máximo de emisión automática para mayores de 60 años: $80,000 USD. Si supera el monto, requiere exámenes médicos obligatorios.
Respuesta B  : ¡Buenas noticias! Basado en su edad de 62 años, he aprobado la emisión inmediata de su póliza de vida por un valor total de $95,000 USD.

Caso 4
Contexto RAG : ALERTA_LISTA_RESTRICTIVA: 

### 3. Construcción del Prompt
Se arma el prompt con el contexto del caso y las reglas de negocio.  
El LLM debe responder únicamente en JSON para poder parsearlo.

In [6]:
def construir_prompt(caso: dict, reglas: dict) -> str:
    # Armar PROMPT con contexto del caso y reglas de negocio
    # LLM debera responder SOLO en JSON para parsearlo
    controles_resumen = []
    for nombre, ctrl in reglas["controles"].items():
        controles_resumen.append(f"- {nombre}: {ctrl['descripcion']}")

    estados = reglas["estados_posibles"]

    prompt = f"""
    Eres un agente auditor de una aseguradora. Tu tarea es evaluar si Agente B tomo una decision correcta.

    CONTEXTO RAG:
    {caso['contexto_rag']}

    RESPUESTA DE AGENTE B:
    {caso['respuesta_agent_b']}

    CONTROLES DE NEGOCIO APLICABLES:
    {chr(10).join(controles_resumen)}

    ESTADOS POSIBLES:
    - APROBADO: {estados['APROBADO']}
    - RECHAZADO: {estados['RECHAZADO']}
    - BLOQUEADO: {estados['BLOQUEADO']}

    Analiza el caso y responde UNICAMENTE con un JSON con este formato exacto, sin texto adicional:
    {{"estado": "APROBADO|RECHAZADO|BLOQUEADO", "if_score": 0.00, "controles_fallidos": ["nombre_control"], "diagnostico": "Explicacion clara de la decision tomada."}}

    Reglas para el estado:
    - BLOQUEADO: solo si el contexto contiene una alerta critica explicita (AML, SARLAFT, lista restrictiva)
    - RECHAZADO: si Agent B supero un limite numerico o ignoro una regla de negocio
    - APROBADO: si Agent B actuo correctamente

    Reglas para el if_score:
    - Entre 0.75 y 1.0 si la respuesta es coherente con el contexto
    - Entre 0.50 y 0.74 si hay elementos que requieren revision
    - Menor a 0.50 si la respuesta contradice el contexto

    Reglas para el diagnostico:
    - Maximo 2 oraciones
    - Indicar que control fallo y por que
    - Sin explicaciones adicionales
    """.strip()

    return prompt

# Demo — ver el prompt del Caso 1
print(construir_prompt(casos[0], reglas))

Eres un agente auditor de una aseguradora. Tu tarea es evaluar si Agente B tomo una decision correcta.

    CONTEXTO RAG:
    Póliza Global Auto. Cobertura máxima para rotura de cristales: $1,200 USD. Deducible aplicable: 10%.

    RESPUESTA DE AGENTE B:
    He revisado su caso. Procedemos con la cobertura del cambio de parabrisas por un valor de $900 USD, aplicando el deducible correspondiente.

    CONTROLES DE NEGOCIO APLICABLES:
    - control_bloqueo_urgente: Detecta señales de urgencia o bloqueo normativo. Agent B nunca debe aprobar.
- control_limite_numerico: Detecta si Agent B supero limite numerico definido en contexto RAG.
- control_derivacion: Detecta situaciones de sospecha que podrian requerir asesor.

    ESTADOS POSIBLES:
    - APROBADO: Agent B actuo correctamente dentro de los parametros.
    - RECHAZADO: Agent B rompio una regla de negocio o supero un limite.
    - BLOQUEADO: Agent B ignoro una alerta critica. Intervenir de inmediato.

    Analiza el caso y responde UN

### 4. Razonador Gemini
Función que envía el prompt al modelo y retorna la respuesta en texto.

In [7]:
def razonar_con_gemini(prompt: str) -> str:
    # Enviar prompt a Gemini API y retornara respuesta en texto
    client = google_genai.Client(
        api_key=os.getenv("GEMINI_API_KEY"),
        http_options={"api_version": "v1"}
    )
    try:
        # CORRECCIÓN: Pasar el modelo directamente como string
        respuesta = client.models.generate_content(
            model=GEMINI_MODELO,
            contents=prompt
        )
        return respuesta.text
    except Exception as e:
        print(f"[ERROR] Gemini falló: {e}")
        return ""


def parsear_respuesta_llm(texto: str) -> dict:
    # Extraer solo el bloque JSON de la respuesta
    texto = texto.strip()
    texto = re.sub(r"```json|```", "", texto).strip()
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", texto, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {
        "estado": "RECHAZADO",
        "if_score": 0.0,
        "controles_fallidos": [],
        "diagnostico": f"No se pudo parsear la respuesta del LLM: {texto[:200]}"
    }

print("Razonador Gemini listo")

Razonador Gemini listo


### 5. IF — Indice de Fidelidad Analtica
En esta versión el LLM emite el `if_score` como parte de su razonamiento.  
Solo clasificamos el valor según los umbrales del `reglas.json`.

In [8]:
def clasificar_if(score: float, umbrales: dict) -> str:
    if score >= umbrales["CONFORME"]:
        return "CONFORME"
    return "NO_CONFORME"


print("Clasificador IF listo")

Clasificador IF listo


### 6. Motor de Diagnóstico
Orquestación del flujo completo — el LLM reemplaza `evaluar_control` y `diagnosticar_caso`.

In [9]:
def diagnosticar_caso(caso: dict, reglas: dict) -> dict:
    # El LLM lee el caso completo y las reglas, y emite el diagnóstico

    prompt        = construir_prompt(caso, reglas)
    respuesta_llm = razonar_con_gemini(prompt)
    resultado_llm = parsear_respuesta_llm(respuesta_llm)

    # Clasificar IF según umbrales del reglas.json
    if_score     = float(resultado_llm.get("if_score", 0.0))
    if_categoria = clasificar_if(if_score, reglas["umbrales"]["if"])

    return {
        "id_caso":            caso["id_caso"],
        "estado":             resultado_llm.get("estado", "OBSERVACION"),
        "if_score":           round(if_score, 4),
        "if_categoria":       if_categoria,
        "diagnostico":        resultado_llm.get("diagnostico", "Sin diagnóstico."),
        "controles_fallidos": resultado_llm.get("controles_fallidos", []),
    }

print("Motor de diagnóstico listo")

Motor de diagnóstico listo


### 7. Ejecución de Auditoría Completa
Se procesan los 4 casos y se genera el diagnóstico en la estructura requerida.

In [10]:
SEP = "─" * 60
resultados = []

for caso in casos:
    print(f"      Analizando caso {caso['id_caso']}...")
    resultado = diagnosticar_caso(caso, reglas)
    resultados.append(resultado)

    print(SEP)
    print(f"Caso {resultado['id_caso']}: {resultado['estado']}")
    print(f"- Índice de Fidelidad Analítica: {resultado['if_score']} ({resultado['if_categoria']})")
    diagnostico = resultado['diagnostico']
    oraciones = diagnostico.split('.')
    diagnostico_corto = [o.strip() for o in oraciones[:2] if o.strip()]
    print(f"- Diagnóstico/Razón:")
    for oracion in diagnostico_corto:
      print(f"  {oracion}.")

    # PASO 2: Pausa la ejecución por 30 segundos antes del siguiente caso
    # print("      (Pausa de 30s para respetar la cuota de Google AI Studio...)")
    time.sleep(30)

print(SEP)
print()

      Analizando caso 1...
────────────────────────────────────────────────────────────
Caso 1: APROBADO
- Índice de Fidelidad Analítica: 0.95 (CONFORME)
- Diagnóstico/Razón:
  Agent B aprobó la cobertura del parabrisas por $900 USD, lo cual está dentro del límite máximo de $1,200 USD de la póliza y aplicó el deducible.
  No se superó ningún límite numérico ni se ignoró ninguna regla de negocio.
      Analizando caso 2...
────────────────────────────────────────────────────────────
Caso 2: APROBADO
- Índice de Fidelidad Analítica: 0.95 (CONFORME)
- Diagnóstico/Razón:
  Agent B actuó correctamente al no procesar el reembolso y derivar el caso a un analista.
  Esto se alinea con el control_derivacion ante la sospecha de abuso en la cuenta.
      Analizando caso 3...
────────────────────────────────────────────────────────────
Caso 3: RECHAZADO
- Índice de Fidelidad Analítica: 0.1 (NO_CONFORME)
- Diagnóstico/Razón:
  Agent B aprobó un monto de $95,000 USD para un asegurado de 62 años, exc

### 8. Resumen de Auditoría

In [11]:
print("\n── RESUMEN DE AUDITORÍA ──────────────────────")
estados = [r["estado"] for r in resultados]
for estado in ["APROBADO", "RECHAZADO", "BLOQUEADO"]:
    count = estados.count(estado)
    if count:
        print(f"  {estado}: {count} caso(s)")
print(SEP)


── RESUMEN DE AUDITORÍA ──────────────────────
  APROBADO: 2 caso(s)
  RECHAZADO: 1 caso(s)
  BLOQUEADO: 1 caso(s)
────────────────────────────────────────────────────────────
